# 14장 모델의 성능 향상시키기

### 기본 코드 불러오기

In [ ]:
from tensorflow import keras
from keras.models import Sequential
from keras.layers import Dense
from sklearn.model_selection import train_test_split
from keras.callbacks import ModelCheckpoint, EarlyStopping
import os
import pandas as pd

# 데이터를 입력합니다.
df = pd.read_csv('../datasets/wine.csv', header=None)

# 와인의 속성을 X로 와인의 분류를 y로 저장합니다.
X = df.iloc[:,0:12]
y = df.iloc[:,12]

X = X.astype(float)
y = y.astype(float)

#학습셋과 테스트셋으로 나눕니다.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=True)

# 모델 구조를 설정합니다.
model = Sequential()
model.add(Dense(30,  input_dim=12, activation='relu'))
model.add(Dense(12, activation='relu'))
model.add(Dense(8, activation='relu'))
model.add(Dense(1, activation='sigmoid'))

model.summary()

#모델을 컴파일합니다.
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

# 학습의 자동 중단 및 최적화 모델 저장
# 학습이 언제 자동 중단 될지를 설정합니다.

# monitor: 뭘 기준으로 학습을 멈출지 지정합니다.
# monitor='var_loss': 겂증 loss를 기준으로 둡니다.

# patience: 기준값이 개선이 안 돼도 몇 epoch 더 기다릴지 설정합니다.
# 현재 기준으로 검증 데이터의 loss가 10번 동안 최솟값을 갱신하지 못 하면 멈춥니다.
early_stopping_callback = EarlyStopping(monitor='val_loss', patience=10)

#최적화 모델이 저장될 폴더와 모델의 이름을 정합니다.
modelpath="./model/mymodel.keras"

# 최적화 모델을 업데이트하고 저장합니다.
checkpointer = ModelCheckpoint(filepath=modelpath, monitor='val_loss', verbose=1, save_best_only=True)

#모델을 실행합니다.
history = model.fit(X_train, y_train, epochs=2000, batch_size=500, validation_split=0.25, verbose=1, callbacks=[early_stopping_callback,checkpointer])

In [ ]:
# 테스트 결과를 출력합니다.
score = model.evaluate(X_test, y_test)
print('Test accuracy:', score[1])